# 13.08 - CLIP prompt ensembling and calibration

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Prompt ensemble and zero-shot calibration report.

This third vision-language lesson practices CLIP's embedding boundary without requiring a checkpoint download. Prepared image and prompt embeddings let you focus on normalization, prompt prototypes, probabilities, and confidence calibration.

## Core Ideas

CLIP compares normalized image and text embeddings with cosine similarity. Multiple prompt templates can be averaged into a class prototype, but the mean should be normalized again. The logit scale changes confidence without changing ranking. Accuracy alone does not reveal whether confidence is trustworthy, so calibration evidence such as expected calibration error (ECE) is useful.

In [ ]:
import numpy as np
import pandas as pd
import torch

SEED = 13
np.random.seed(SEED)
torch.manual_seed(SEED)

## Prepared Offline Embeddings

There are three classes, three prompt templates per class, and twelve images. Values imitate cached encoder outputs and keep the lesson deterministic and competition-safe.

In [ ]:
base_text = torch.tensor([[1.0, 0.0, 0.1, 0.0], [0.0, 1.0, 0.0, 0.1], [0.1, 0.0, 1.0, 0.0]], dtype=torch.float32)
prompt_offsets = torch.tensor([[0.00, 0.00, 0.00, 0.00], [0.04, -0.03, 0.02, 0.00], [-0.03, 0.02, 0.00, 0.03]], dtype=torch.float32)
prompt_embeddings = base_text[:, None, :] + prompt_offsets[None, :, :]
image_labels = torch.arange(3).repeat_interleave(4)
image_embeddings = base_text[image_labels] + 0.12 * torch.randn(12, 4)
print("image/prompts:", image_embeddings.shape, prompt_embeddings.shape, "support:", torch.bincount(image_labels).tolist())

## Exercise 13-A: Normalize encoder outputs

Reject zero vectors rather than allowing undefined cosine similarities.

**Return structure — `l2_normalize`:** A CPU `torch.float32` tensor with the same shape as the input and unit L2 norm along its final dimension.

In [ ]:
# TODO 13-A
def l2_normalize(embeddings, eps=1e-12):
    raise NotImplementedError("Complete Exercise 13-A")


# Smoke check: normalize every prepared image embedding.
normalized_images = l2_normalize(image_embeddings)
print("image norms:", normalized_images.norm(dim=-1)[:4])

## Exercise 13-B: Ensemble prompt templates

Average the normalized templates within each class, then normalize the resulting class prototype.

**Return structure — `ensemble_prompt_prototypes`:** A CPU float tensor `[C,D]` for input `[C,P,D]`, where `C` is classes, `P` prompts per class, and every output row has unit norm.

In [ ]:
# TODO 13-B
def ensemble_prompt_prototypes(class_prompt_embeddings):
    raise NotImplementedError("Complete Exercise 13-B")


# Smoke check: build three class prototypes.
ensemble_prototypes = ensemble_prompt_prototypes(prompt_embeddings)
single_prototypes = l2_normalize(prompt_embeddings[:, 0, :])
print("prototypes:", ensemble_prototypes.shape)

## Exercise 13-C: Calculate zero-shot probabilities

Similarity logits are `logit_scale × image @ class.T`. Softmax is applied across classes.

**Return structure — `zero_shot_probabilities`:** A CPU `torch.float32` tensor `[N,C]`. Every row is non-negative and sums to approximately 1.

In [ ]:
# TODO 13-C
def zero_shot_probabilities(images, class_prototypes, logit_scale=10.0):
    raise NotImplementedError("Complete Exercise 13-C")


# Smoke check: calculate single-prompt and ensemble probabilities.
single_probabilities = zero_shot_probabilities(image_embeddings, single_prototypes)
ensemble_probabilities = zero_shot_probabilities(image_embeddings, ensemble_prototypes)
print("probability rows:", ensemble_probabilities[:3])

## Exercise 13-D: Measure expected calibration error

Split confidence into equal-width bins and weight each bin's absolute accuracy–confidence gap by its sample share.

**Return structure — `expected_calibration_error`:** A Python `float` in `[0,1]`. `probabilities` has shape `[N,C]` and `labels` has shape `[N]`.

In [ ]:
# TODO 13-D
def expected_calibration_error(probabilities, labels, n_bins=5):
    raise NotImplementedError("Complete Exercise 13-D")


# Smoke check: measure ensemble calibration.
ensemble_ece = expected_calibration_error(ensemble_probabilities, image_labels)
print("ensemble ECE:", ensemble_ece)

## Exercise 13-E: Compare prompt strategies

Expose accuracy, confidence, ECE, and change from the first strategy in one evidence table.

**Return structure — `prompt_strategy_table`:** A `pandas.DataFrame` with columns `strategy`, `sample_count`, `class_support`, `accuracy`, `mean_confidence`, `ece`, and `delta_accuracy`.

In [ ]:
# TODO 13-E
def prompt_strategy_table(named_probabilities, labels):
    raise NotImplementedError("Complete Exercise 13-E")


# Smoke check and full prepared-set evidence.
prompt_evidence = prompt_strategy_table({"single_prompt": single_probabilities, "ensemble": ensemble_probabilities}, image_labels)
print(prompt_evidence.to_string(index=False))

## Test Cases

**Return structure — `run_day13_tests`:** Returns `None`; assertions and `Day 13 tests passed` communicate success.

In [ ]:
def run_day13_tests():
    assert normalized_images.shape == (12, 4) and normalized_images.dtype == torch.float32
    assert torch.allclose(normalized_images.norm(dim=1), torch.ones(12), atol=1e-5)
    assert ensemble_prototypes.shape == (3, 4)
    assert ensemble_probabilities.shape == (12, 3)
    assert torch.allclose(ensemble_probabilities.sum(dim=1), torch.ones(12), atol=1e-6)
    assert 0.0 <= ensemble_ece <= 1.0
    assert list(prompt_evidence.columns) == ["strategy", "sample_count", "class_support", "accuracy", "mean_confidence", "ece", "delta_accuracy"]
    assert prompt_evidence["sample_count"].tolist() == [12, 12]
    assert all(support == [4, 4, 4] for support in prompt_evidence["class_support"])
    print("Day 13 tests passed")


run_day13_tests()

## Day 13 Checklist

- [ ] Normalize image and text embeddings.
- [ ] Normalize prompt templates before and after averaging.
- [ ] Explain how logit scale changes confidence.
- [ ] Compare accuracy and ECE on the same labeled fixture.
- [ ] Run the test cases.